<a href="https://colab.research.google.com/github/NhanChanDeo/VQA_Final_Thesis/blob/main/DATASETS%20UPLOAD/Step2_3(1)_Fetch_Extract_DocVQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Steps 2 & 3: Fetch DocVQA to Colab, Extract Archives, and Sync to Hugging Face Hub

This notebook executes the complete ingestion, extraction, and sync pipeline for **DocVQA Task 1 (Single Page Document VQA)**:
1. **High-Speed Download:** Fetches document images (`spdocvqa_images.tar.gz`), questions and answers (`spdocvqa_qas.zip`), OCR outputs (`spdocvqa_ocr.tar.gz`), and preprocessed IMDBs (`spdocvqa_imdb.zip`) using `aria2c`.
2. **Canonical Structuring & Sequential Decompression:** Unpacks all archives into their respective uncompressed folders (`images/`, `annotations/`, `ocr/`, `imdb/`), immediately deleting local archives after decompression to optimize VM disk space.
3. **Dataset Card Generation:** Updates `README.md` to reflect the uncompressed directory structure and metadata.
4. **Hugging Face Hub Sync:** Pushes the uncompressed directory via `HfApi.upload_folder(...)` with `delete_patterns=["*.tar.gz", "*.zip"]` to replace old remote archives with native files.
5. **Disk Reclaim:** Purges local files to keep Colab VM storage clean.

In [2]:
# 1. Setup tools and authenticate
!sudo apt-get install -y -qq aria2
!pip install -q --upgrade huggingface_hub

import os
import shutil
from pathlib import Path
from huggingface_hub import HfApi, login

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, ModuleNotFoundError):
    HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found! Please configure HF_TOKEN in Colab Secrets.")

login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)
username = api.whoami()['name']
DOCVQA_REPO = f"{username}/docvqa-task1-spdocvqa"
print(f"âœ“ Authenticated as: {username}")
print(f"âœ“ Target Repo: {DOCVQA_REPO}")

âœ“ Authenticated as: gamusa
âœ“ Target Repo: gamusa/docvqa-task1-spdocvqa


In [3]:
# 2. Check initial disk allocation
!df -h /

Filesystem      Size  Used Avail Use% Mounted on
overlay         226G   21G  206G  10% /


In [4]:
# 3. Download DocVQA Task 1 files via aria2c
DOCVQA_RAW = Path("./docvqa_raw")
DOCVQA_CLEAN = Path("./docvqa_dataset")

shutil.rmtree(DOCVQA_RAW, ignore_errors=True)
shutil.rmtree(DOCVQA_CLEAN, ignore_errors=True)
DOCVQA_RAW.mkdir(parents=True, exist_ok=True)

# Direct backend download URLs from CVC / RRC Challenge 17
docvqa_urls = """
https://datasets.cvc.uab.es/rrc/DocVQA/Task1/spdocvqa_qas.zip
https://datasets.cvc.uab.es/rrc/DocVQA/Task1/spdocvqa_images.tar.gz
https://datasets.cvc.uab.es/rrc/DocVQA/Task1/spdocvqa_ocr.tar.gz
https://datasets.cvc.uab.es/rrc/DocVQA/Task1/spdocvqa_imdb.zip
"""

with open("docvqa_urls.txt", "w") as f:
    f.write(docvqa_urls.strip())

print(">>> Starting high-speed parallel download with aria2c...")
!aria2c -j 4 -x 8 -s 8 -k 1M --check-certificate=false -i docvqa_urls.txt --dir={DOCVQA_RAW}

print("Downloaded raw assets:")
!ls -lh {DOCVQA_RAW}

>>> Starting high-speed parallel download with aria2c...

10/01 04:42:32 [NOTICE] Downloading 4 item(s)

10/01 04:42:37 [NOTICE] Download complete: docvqa_raw/spdocvqa_qas.zip

10/01 04:43:05 [NOTICE] Download complete: docvqa_raw/spdocvqa_ocr.tar.gz
 *** Download Progress Summary as of Thu Oct  1 04:43:33 2026 *** 
=
[#94ac96 136MiB/7.9GiB(1%) CN:8 DL:2.8MiB ETA:47m18s]
FILE: docvqa_raw/spdocvqa_images.tar.gz
-
[#a12558 131MiB/303MiB(43%) CN:8 DL:2.6MiB ETA:1m3s]
FILE: docvqa_raw/spdocvqa_imdb.zip
-

 *** Download Progress Summary as of Thu Oct  1 04:44:33 2026 *** 
=
[#94ac96 310MiB/7.9GiB(3%) CN:8 DL:2.6MiB ETA:49m10s]
FILE: docvqa_raw/spdocvqa_images.tar.gz
-
[#a12558 298MiB/303MiB(98%) CN:8 DL:2.9MiB ETA:1s]
FILE: docvqa_raw/spdocvqa_imdb.zip
-


10/01 04:44:36 [NOTICE] Download complete: docvqa_raw/spdocvqa_imdb.zip
 *** Download Progress Summary as of Thu Oct  1 04:45:34 2026 *** 
=
[#94ac96 633MiB/7.9GiB(7%) CN:8 DL:5.5MiB ETA:22m49s]
FILE: docvqa_raw/spdocvqa_images.tar.gz
-



In [ ]:
# 4. Restructure & Extract all archives into uncompressed folders
import tarfile
import zipfile

(DOCVQA_CLEAN / "images").mkdir(parents=True, exist_ok=True)
(DOCVQA_CLEAN / "annotations").mkdir(parents=True, exist_ok=True)
(DOCVQA_CLEAN / "ocr").mkdir(parents=True, exist_ok=True)
(DOCVQA_CLEAN / "imdb").mkdir(parents=True, exist_ok=True)

# Move downloaded archives into destination directories
for file in DOCVQA_RAW.iterdir():
    if "images" in file.name:
        shutil.move(str(file), DOCVQA_CLEAN / "images" / file.name)
    elif "qas" in file.name:
        shutil.move(str(file), DOCVQA_CLEAN / "annotations" / file.name)
    elif "ocr" in file.name:
        shutil.move(str(file), DOCVQA_CLEAN / "ocr" / file.name)
    elif "imdb" in file.name:
        shutil.move(str(file), DOCVQA_CLEAN / "imdb" / file.name)

# 4.1 Extract Annotations (QAs)
qas_zip = DOCVQA_CLEAN / "annotations" / "spdocvqa_qas.zip"
if qas_zip.exists():
    print(">>> Extracting annotations...")
    with zipfile.ZipFile(qas_zip, "r") as z:
        z.extractall(DOCVQA_CLEAN / "annotations")
    qas_zip.unlink()
    print("✓ Annotations extracted & zip removed")

# 4.2 Extract OCR tokens
ocr_tar = DOCVQA_CLEAN / "ocr" / "spdocvqa_ocr.tar.gz"
if ocr_tar.exists():
    print(">>> Extracting OCR tokens...")
    with tarfile.open(ocr_tar, "r:gz") as tar:
        tar.extractall(DOCVQA_CLEAN / "ocr")
    ocr_tar.unlink()
    print("✓ OCR tokens extracted & tar.gz removed")

# 4.3 Extract IMDB indices
imdb_zip = DOCVQA_CLEAN / "imdb" / "spdocvqa_imdb.zip"
if imdb_zip.exists():
    print(">>> Extracting IMDB files...")
    with zipfile.ZipFile(imdb_zip, "r") as z:
        z.extractall(DOCVQA_CLEAN / "imdb")
    imdb_zip.unlink()
    print("✓ IMDB files extracted & zip removed")

# 4.4 Extract Images (~12,767 files - takes 2-4 minutes)
images_tar = DOCVQA_CLEAN / "images" / "spdocvqa_images.tar.gz"
if images_tar.exists():
    print(">>> Extracting document images...")
    with tarfile.open(images_tar, "r:gz") as tar:
        tar.extractall(DOCVQA_CLEAN / "images")
    images_tar.unlink()
    print("✓ Document images extracted & tar.gz removed")

# Dataset Card
docvqa_readme = """---
license: other
task_categories:
- visual-question-answering
size_categories:
- 10K<n<100K
tags:
- docvqa
- document-ai
- vqa
- multimodal
---

# DocVQA (Task 1: Single Page Document VQA)

- **Source Portal:** [Robust Reading Competition (RRC) Challenge 17](https://rrc.cvc.uab.es/?ch=17&com=downloads)
- **Host:** Computer Vision Center (CVC) / IIIT Hyderabad (UCSF Industry Documents Library)
- **Terms:** Hosted purely for non-commercial research and educational purposes.

## Repository Contents (Extracted)
- `images/`: 12,767 uncompressed document images (.png/.jpg)
- `annotations/`: Uncompressed train, val, and test JSON QA pairs (~50,000 questions)
- `ocr/`: Microsoft OCR token annotations (word bounding boxes and transcriptions)
- `imdb/`: Uncompressed Image Database (IMDB) files for MP-DocVQA
"""

with open(DOCVQA_CLEAN / "README.md", "w", encoding="utf-8") as f:
    f.write(docvqa_readme.strip())

print("\nStructured extracted directory:")
!ls -lh {DOCVQA_CLEAN}


In [ ]:
# 5. Sync extracted dataset to Hugging Face Hub (deletes old archives remotely)
print(f">>> Uploading extracted DocVQA dataset to https://huggingface.co/datasets/{DOCVQA_REPO}...")
api.upload_folder(
    folder_path=str(DOCVQA_CLEAN),
    repo_id=DOCVQA_REPO,
    repo_type="dataset",
    commit_message="Extract all archives and sync uncompressed files with folder structure",
    delete_patterns=["*.tar.gz", "*.zip"],  # Purges old archives from Hugging Face repo
)
print("✓ Upload and extraction sync completed successfully!")

In [ ]:
# 6. Verify remote files on Hugging Face Hub
print(f"=== Files in https://huggingface.co/datasets/{DOCVQA_REPO} ===")
remote_files = api.list_repo_files(repo_id=DOCVQA_REPO, repo_type="dataset")
for f in remote_files:
    print(f" - {f}")

In [ ]:
# 7. Purge local disk to reclaim space
print(">>> Purging local files...")
shutil.rmtree(DOCVQA_RAW, ignore_errors=True)
shutil.rmtree(DOCVQA_CLEAN, ignore_errors=True)
if os.path.exists("docvqa_urls.txt"):
    os.remove("docvqa_urls.txt")

print("Remaining disk space after purge:")
!df -h /